Задание:

Реализовать предсказание настроений с помощью классических алгоритмов + tf-idf на основе данных твиттов и наивным байесом.

На "отлично" - попробовать обучить любой из пройденных алгоритмов на основе эмбеддингов взвешенных по attention или с помощью классических от TruncatedSVD на основе полиномиальных от tf-idf с (1-3) биграммами.

Поскольку глубокие сети еще впереди код конвертации списка предложений в эмбеддинг по ссылке выше.

Так как мы чаще всего работаем с нампай массивами в обучении, используйте:

sentence_embeddings.detach().numpy()

## Загрузка 

In [2]:
# import kagglehub
# path = kagglehub.dataset_download("arkhoshghalb/twitter-sentiment-analysis-hatred-speech", output_dir='../data/dataset_text')

100%|█████████████████████████████████████████████████████████████████████████████████████| 1.89M/1.89M [00:00<00:00, 2.17MB/s]

Extracting files...


In [2]:
import numpy as np
import pandas as pd

import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

import warnings
warnings.simplefilter('ignore')

In [3]:
df_text = pd.read_csv('data/dataset_text/train.csv')
df_text_test = pd.read_csv('data/dataset_text/test.csv')

In [48]:
df_text.head()

,id,label,tweet
0,1,0,@user when a father is dysfunctional and is s...
1,2,0,@user @user thanks for #lyft credit i can't us...
2,3,0,bihday your majesty
3,4,0,#model i love u take with u all the time in ...
4,5,0,factsguide: society now #motivation


In [5]:
df_text.groupby('label').count()

,id,tweet
label,,
0,29720,29720
1,2242,2242


## План

1. Видим дисбаланс - попробовать методы для приведения к нормальному балансу классов:

   * Семплирование - оверсемплинг: EDA
  
   * Smote 
  
   * Также попробовать сбалансировать удалив элементы мажоритарного класса.

   * Аугментацией после перевода текста в векторы.

2. Все тексты токенизовароть и использовать методы лемматизации и стемминга.

3. Выкинуть стоп слова - самые частые.

4. Собрать мешок слов. Так же мешок слов n-граммами (1, 2, 3)

5. TF-IDf

6. Предложенный обученный (S)BERT

## Семплинг 

### Оверсемплинг

EDA - создаем дополнительные примеры недостающего класса 4 методами: замена синонимами, вставка синонима, перестановка пары слов, удаление слов.

Для этого буду использовать библиотеку textaugment. Для предобработки текста (стемминг и лемматизация) и словаря синонимов - библиотеку NTLK

In [9]:
import nltk
import string
import re

from textaugment import EDA
from nltk.corpus import stopwords 
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [36]:
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /Users/mihail/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mihail/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mihail/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [13]:
from textaugment import Wordnet

In [40]:
stop_words = stopwords.words('english')
lemmatizer = WordNetLemmatizer()
def preprocessor(raw_text):
    text = raw_text.lower()

    pattern = f'[{re.escape(string.punctuation)}]'
    text = re.sub(pattern, ' ', text)

    tokens = word_tokenize(text)

    clean_tokens = []

    for word in tokens:
        if word not in stop_words:
            clean_tokens.append(lemmatizer.lemmatize(word))

    return ' '.join(clean_tokens)

In [58]:
df_clean_text = pd.concat([df_text['label'], df_text['tweet'].apply(preprocessor)], axis=1)

In [59]:
def augment_text(df_texts, alpha=0.1, n=3, method='sr'):
    '''
    method sr - synonym replacement
        rd - random deletion
        rw - random swap 
        rs - random insertion
    '''
    aug_text = []
    
    
            

,label,tweet
0,0,user father dysfunctional selfish drag kid dys...
1,0,user user thanks lyft credit use cause offer w...
2,0,bihday majesty
3,0,model love u take u time urð± ðððð...
4,0,factsguide society motivation
...,...,...
31957,0,ate user isz youuu ððððððð...
31958,0,see nina turner airwave trying wrap mantle gen...
31959,0,listening sad song monday morning otw work sad
31960,1,user sikh temple vandalised calgary wso condem...
